In [1]:
from sentence_transformers import SentenceTransformer, InputExample, losses, evaluation
from torch.utils.data import DataLoader
import torch
import pandas as pd
from sklearn.model_selection import train_test_split

SEED = 42
torch.manual_seed(SEED)

In [2]:
import torch
from torch.utils.data import Dataset
import numpy as np
import regex as re

class SBERTDataset(Dataset):
    def __init__(self, dataframe, use_normalized_score=True):
        # convert to list
        self.reference_answers = dataframe['reference_answer'].tolist()
        self.student_answers = dataframe['answer'].tolist() 
        
        # Use either normalized or raw score based on parameter
        if use_normalized_score:
            self.scores = dataframe['normalized_score'].values.astype(np.float32)
        else:
            self.scores = dataframe['score'].values.astype(np.float32)
    
    def preprocess_text(self, text):
        # Remove extra whitespace
        text = ' '.join(text.split())
        # Convert to lowercase
        text = text.lower()
        # Remove special characters (keep punctuation)
        text = re.sub(r'[^a-zA-Z0-9\s.,!?]', '', text)
        return text
    
    def __len__(self):
        return len(self.scores)
    
    def __getitem__(self, idx):
        return {
            'reference_answer': self.preprocess_text(self.reference_answers[idx]),
            'student_answer': self.preprocess_text(self.student_answers[idx]),
            'score': torch.tensor(self.scores[idx], dtype=torch.float)
        }

In [3]:
df = pd.read_csv("../data/aes_dataset_5k_clean.csv")
df = df[df['dataset'] == 'analisis_essay'][['reference_answer', 'answer', 'score', 'normalized_score', 'dataset', 'dataset_num']]
print(df.info())
df.head()

<class 'pandas.core.frame.DataFrame'>
Index: 2162 entries, 0 to 2161
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   reference_answer  2162 non-null   object 
 1   answer            2162 non-null   object 
 2   score             2162 non-null   float64
 3   normalized_score  2162 non-null   float64
 4   dataset           2162 non-null   object 
 5   dataset_num       2162 non-null   object 
dtypes: float64(2), object(4)
memory usage: 118.2+ KB
None


,reference_answer,answer,score,normalized_score,dataset,dataset_num
0,Fungsi karbohidrat adalah sebagai pemasok ener...,"sumber tenaga, pemanis alami, menjaga sistem i...",27.0,0.27,analisis_essay,analisis_essay-1
1,Fungsi karbohidrat adalah sebagai pemasok ener...,"sebagai sumber energi, pemanis alami, menjaga ...",21.0,0.21,analisis_essay,analisis_essay-1
2,Fungsi karbohidrat adalah sebagai pemasok ener...,1. Sebagai energi. 2. Sebagai memperlancaar pe...,42.0,0.42,analisis_essay,analisis_essay-1
3,Fungsi karbohidrat adalah sebagai pemasok ener...,"untuk membuat kenyang, agar tidak lapar, agar ...",18.0,0.18,analisis_essay,analisis_essay-1
4,Fungsi karbohidrat adalah sebagai pemasok ener...,Karbohidrat mempunyai peran penting untuk pros...,82.0,0.82,analisis_essay,analisis_essay-1


In [4]:
def split_dataset(df, train_ratio, valid_ratio, test_ratio):
    print("run split dataset...")
    subset_dataset = df['dataset_num'].unique()
    splits = {}
    for subset in subset_dataset:
        # get data by dataset_num
        subset_df = df[df['dataset_num'] == subset]

        # split dataset
        train_df, temp_df = train_test_split(subset_df, test_size=(1 - train_ratio), random_state=SEED, shuffle=True)
        valid_df, test_df = train_test_split(temp_df, test_size=test_ratio / (valid_ratio + test_ratio), random_state=SEED, shuffle=True)

        # save split dataset
        splits[subset] = {
            'train': train_df,
            'valid': valid_df,
            'test': test_df,
        }
    
    train_dataset = pd.concat([splits[subset]['train'] for subset in subset_dataset])
    valid_dataset = pd.concat([splits[subset]['valid'] for subset in subset_dataset])
    test_dataset = pd.concat([splits[subset]['test'] for subset in subset_dataset])

    return train_dataset, valid_dataset, test_dataset

In [5]:
train, valid, test = split_dataset(df, 0.8, 0.1, 0.1)
train_data = SBERTDataset(train)
valid_data = SBERTDataset(valid)
test_data = SBERTDataset(test)
print(len(train_data))
print(len(valid_data))
print(len(test_data))

run split dataset...
1711
213
238


In [6]:
def create_data(data):    
    data_examples = []
    for item in data:
        score = float(item['score']) if torch.is_tensor(item['score']) else item['score']
        # Create example where the label is the score
        data_examples.append(InputExample(
            texts=[item['reference_answer'], item['student_answer']], 
            label=score
        ))

    return data_examples

train_examples = create_data(train_data)
valid_examples = create_data(valid_data)
test_examples = create_data(test_data)

In [7]:
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=16)
valid_dataloader = DataLoader(valid_examples, shuffle=False, batch_size=16)
test_dataloader = DataLoader(test_examples, shuffle=False, batch_size=16)

In [9]:
import torch
import torch.nn as nn
from transformers import AutoModel, AutoTokenizer
import torch.nn.functional as F

class SiameseModel(nn.Module):
    def __init__(self, model_name='sentence-transformers/paraphrase-multilingual-mpnet-base-v2', dropout=0.1):
        super(SiameseModel, self).__init__()
        
        # Load the model and tokenizer
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.encoder = AutoModel.from_pretrained(model_name)
        
        # Get embedding dimension from the model config
        self.embedding_dim = self.encoder.config.hidden_size
    
    def mean_pooling(self, model_output, attention_mask):
        # Mean pooling - take average of all token embeddings
        token_embeddings = model_output[0]
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
        return torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)
    
    def get_embeddings(self, texts):
        # Tokenize the input texts
        encoded_input = self.tokenizer(
            texts, 
            padding=True, 
            truncation=True, 
            max_length=512, 
            return_tensors='pt'
        )
        
        # Move to the same device as the model
        device = next(self.parameters()).device
        encoded_input = {k: v.to(device) for k, v in encoded_input.items()}
        
        # Get model output (without torch.no_grad to allow fine-tuning)
        outputs = self.encoder(**encoded_input)
        
        # Apply mean pooling to get sentence embeddings
        embeddings = self.mean_pooling(outputs, encoded_input['attention_mask'])
        return embeddings
    
    def forward(self, reference_texts, student_texts):
        # Get embeddings for reference and student texts
        reference_embeddings = self.get_embeddings(reference_texts)
        student_embeddings = self.get_embeddings(student_texts)

        # Normalize embeddings
        ref_embedding = F.normalize(reference_embeddings, p=2, dim=1)
        student_embedding = F.normalize(student_embeddings, p=2, dim=1)
        
        # Compute similarity score
        similarity = torch.sum(ref_embedding * student_embedding, dim=1)
        
        return similarity
    
model = SiameseModel()

In [24]:
text = "aku pergi ke pasar"
student_text = "aku pergi ke supermarket"
embedding = model.get_embeddings(text)
student_embedding = model.get_embeddings(student_text)
print("sim", torch.sum(embedding * student_embedding, dim=1))
embedding

sim tensor([7.4131], grad_fn=<SumBackward1>)


tensor([[ 2.4808e-02,  6.5573e-02, -1.0175e-02,  2.6186e-02,  1.6225e-01,
          4.7819e-02,  5.4686e-02, -1.0385e-01,  1.7497e-01, -3.0809e-03,
          2.7565e-02,  1.8210e-01,  3.5639e-02,  4.4759e-02,  2.3818e-01,
         -8.1664e-02,  9.7713e-02, -2.7663e-02, -1.8459e-01,  1.2642e-01,
         -2.7627e-02, -1.6667e-02,  1.4160e-01,  9.9262e-02, -4.7163e-02,
         -5.7920e-02, -4.0641e-02,  9.8955e-02,  1.8800e-01, -2.9154e-02,
          1.6333e-01, -1.1490e-01,  1.7850e-01, -1.6704e-02,  9.1600e-02,
         -3.2927e-03,  2.8632e-02, -1.0350e-01, -2.8279e-01, -3.6220e-02,
          9.8477e-02, -5.5328e-03, -2.7448e-02,  3.1424e-03, -5.6446e-02,
         -7.8784e-02,  1.1020e-02, -2.5388e-02, -1.1057e-01,  4.2287e-02,
          6.1043e-02, -2.9425e-02, -4.8525e-02,  8.2190e-02,  1.6441e-01,
         -5.2099e-02,  3.4586e-02, -1.0947e-02,  1.1437e-01,  3.7368e-03,
         -2.3334e-02, -1.4071e-01, -4.6595e-02,  8.7882e-02, -4.1305e-02,
         -3.5212e-03,  3.9898e-02, -1.

In [25]:
embedding = F.normalize(embedding, p=2, dim=1)
student_embedding = F.normalize(student_embedding, p=2, dim=1)
print("sim", torch.sum(embedding * student_embedding, dim=1))
embedding

sim tensor([0.8608], grad_fn=<SumBackward1>)


tensor([[ 8.8558e-03,  2.3407e-02, -3.6320e-03,  9.3476e-03,  5.7918e-02,
          1.7070e-02,  1.9521e-02, -3.7070e-02,  6.2460e-02, -1.0998e-03,
          9.8400e-03,  6.5003e-02,  1.2722e-02,  1.5978e-02,  8.5023e-02,
         -2.9151e-02,  3.4880e-02, -9.8747e-03, -6.5892e-02,  4.5126e-02,
         -9.8619e-03, -5.9496e-03,  5.0547e-02,  3.5433e-02, -1.6836e-02,
         -2.0675e-02, -1.4507e-02,  3.5324e-02,  6.7109e-02, -1.0407e-02,
          5.8305e-02, -4.1015e-02,  6.3719e-02, -5.9629e-03,  3.2698e-02,
         -1.1754e-03,  1.0221e-02, -3.6945e-02, -1.0095e-01, -1.2930e-02,
          3.5153e-02, -1.9750e-03, -9.7982e-03,  1.1217e-03, -2.0149e-02,
         -2.8124e-02,  3.9340e-03, -9.0628e-03, -3.9472e-02,  1.5095e-02,
          2.1791e-02, -1.0504e-02, -1.7322e-02,  2.9339e-02,  5.8688e-02,
         -1.8598e-02,  1.2346e-02, -3.9076e-03,  4.0828e-02,  1.3339e-03,
         -8.3293e-03, -5.0230e-02, -1.6633e-02,  3.1371e-02, -1.4745e-02,
         -1.2570e-03,  1.4242e-02, -4.